In [59]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
import time
import pyperclip

class LOCAL_API():
    def __init__(self):
        self.history = {'last_prompt_no': -1, 'last_response_id': None}
        self.__init__driver()
        
    def create_web_driver(self, position_x=1920, position_y=0, size_x=1400, size_y=600):
        options = webdriver.ChromeOptions()
        # Set Chrome options to block images, CSS, fonts, etc.
        chrome_prefs = {
            "profile.managed_default_content_settings.images": 2,
            "profile.managed_default_content_settings.stylesheets": 2,
            "profile.managed_default_content_settings.fonts": 2,
            "profile.managed_default_content_settings.plugins": 2,
            "profile.managed_default_content_settings.popups": 2,
            "profile.managed_default_content_settings.geolocation": 2,
            "profile.managed_default_content_settings.notifications": 2,
        }
        options.add_experimental_option("prefs", chrome_prefs)
        # options.add_argument("--headless")  # Run headless if you don't need a visible browser
        # options.add_argument('--start-maximized')  # maximized window
        # Set window size: width=800px, height=600px
        options.add_argument(f"--window-size={size_x},{size_y}")
        # Set window position: x=100px from left, y=50px from top
        options.add_argument(f"--window-position={position_x},{position_y}")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")

        # Initialize driver
        # return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return webdriver.Chrome(service=Service(r"C:\Chrome_Driver\chromedriver.exe"), options=options)
    
    def __init__driver(self, delay=0.5):
        self.driver = self.create_web_driver()
        self.creation_time = time.time()
        self.driver.get("https://gemini.google.com/app")
        time.sleep(delay)  # wait for page to load

    def quit_driver(self):
        self.driver.quit()
    
    def restart_driver(self):
        try:
            self.quit_driver()
        except: pass
        self.__init__driver()
    
    def clear_history(self):
        self.history = {'last_prompt_no': -1, 'last_response_id': None}

    def get_prompt_box(self):
        return self.driver.find_element(By.XPATH, '//div[@role="textbox" and @aria-label="Enter a prompt here"]')

    def get_send_button(self):
        return self.driver.find_element(By.XPATH, '//button[contains(@class, "send-button")]')# Click the button

    def send_prompt(self, msg, delay=0.01):
        # Locate the element using XPath
        try:
            text_box = self.get_prompt_box()
            self.driver.execute_script("arguments[0].textContent = arguments[1];", text_box, msg.rstrip('\n'))
        except Exception as e:
            return f"Error occurred while sending prompt: {e}"
        # print("Prompt sent to box") # delete later
        # time.sleep(delay)  ## no need to wait here
        err = "Button not found: timeout"
        start_time = time.time()
        while (time.time() - start_time) < 3:  # timeout after 3 seconds
            # print("Trying to find button") # delete later
            try:
                button = self.get_send_button()
                label = button.get_attribute("aria-label")
                if "Send" in label:
                    button.click()
                    self.history['last_prompt_no'] += 1
                    self.history[self.history['last_prompt_no']] = {'prompt': msg, 'response': None}
                    return "Prompt sent and button clicked successfully"
                # elif "Stop" in label:
                #     print("This is the Stop button")
            except Exception as e:
                err = e
            time.sleep(0.1)  # Wait before trying again
        return f"Error occurred while clicking button: {err}"
    
    def get_last_new_response(self):  
        err, div_text, new_response_id = None, None, False
        try:
            button = self.get_send_button()
            label = button.get_attribute("aria-label")
            if "Stop" in label:
                return {'response': None, 'error': 'still generating', 'id': None}
        except:
            pass

        try: 
            response_containers = self.driver.find_elements(    # each container is a prompt + response group
                By.XPATH, 
                "//div[starts-with(@class, 'conversation-container')]" # old = 'conversation-container' conversation-container
            )
            print(f"Found {len(response_containers)} response containers")  # debug line
            if response_containers:
                new_response_id = response_containers[-1].get_attribute("id")
                if new_response_id == self.history['last_response_id'] and self.history['last_response_id']:
                    return {'response': None, 'error': 'no new response', 'id': None}
                
                div_elements = response_containers[-1].find_elements( # each element is a response of a prompt
                    By.XPATH,
                    ".//message-content[starts-with(@id,'message-content')]"  # old:   @class,'model-response-text'
                )
                if div_elements:
                    div_text = div_elements[-1].text
                else:
                    err = "No response text found"
            else:
                err = "No response containers found"
        except Exception as e:
            err = e
        response = {'response': div_text, 'error': err, 'id':  new_response_id if new_response_id else None}
        if new_response_id:
            self.history[self.history['last_prompt_no']]['response'] = response
            self.history['last_response_id'] = new_response_id
        return response
    
    def copy_last_response_to_clipboard(self):
        pass
        # steps: same as get_last_new_response, then find the copy button and click it
        # try:
        #     <button _ngcontent-ng-c4159091283 mat-button tabindex="0" mattooltip="Copy response" aria-label="Copy" data-test-id="copy-button" class="mdc-button mat-mdc-button-base mat-mdc-tooltip-trigger icon-button mat-mdc-button mat-unthemed" mat-ripple-loader-class-name="mat-mdc-button-ripple" jslog="178035;track:generic_click,impression;BardVeMetadataKey:[["r_7711edcb96c84e21","c_a39913e6d4337352",null,"rc_c5681538456025e4",null,null,"en",null,1,null,null,1,0]];mutable:true" aria-describedby="cdk-describedby-message-ng-1-11" cdk-describedby-host="ng-1">flex
        # except Exception as e:
        #     return f"Error occurred while copying to clipboard: {e}"

    def execute_prompt(self, prompt, inertial_delay=0.4, timeout=30):
        send_status = self.send_prompt(prompt)
        response = {'response': None, 'error': None, 'id': None}
        result = {'prompt': prompt, 'response': response}
        # print(send_status)
        if "Error" in send_status:
            result['response']['error'] = send_status
            return result
        
        time.sleep(inertial_delay)
        result['response']['error'] = "timeout after sending prompt"
        start_time = time.time()
        while (time.time() - start_time) < timeout:
            response = self.get_last_new_response()
            if response['error'] == 'still generating':
                time.sleep(0.2)
                continue
            elif response['error'] == 'no new response':
                time.sleep(0.2)
                continue
            else:
                result['response'] = response
                break
        return result

    def execute_prompts(self, prompts, inertial_delay=0.4, timeout=30, delay_between_prompts=0.4):
        results = []
        for prompt in prompts:
            result = self.execute_prompt(prompt, inertial_delay, timeout)
            results.append(result)
            if result['response']['error'] and 'timeout' in result['response']['error']:
                break
            time.sleep(delay_between_prompts)
        return results


In [60]:
api = LOCAL_API()
# api.execute_prompt("what is the result of 1094*5678?\n response like 'ans = <number>'")

In [3]:
prompt = """
translate the questions to english.
write latex code where necessary (i.e. formula of compound [Cr(NH_3)_6]^{3+}). 
write all text in code editor 
insert subscript "_" where needed. (i.e. H_2O)
To keep a space between value and unit insert "\," between. i.e. mass = 400\,g and velocity = 30\,kmh^{-1} and (3,\,0)

write a,b,c,d options in 1 line when options lengths are small
write a,b,c,d options in 2 lines (2 per line) when option length are big

sample output format/style to follow:
01. Which of the following is diamagnetic? 
a. [Cr(NH_3)_6]^{3+}	b. [Ni(CN)_4]^{2-}  
c. [CuCl_4]^{2-}  	d. [CoF_6]^{2-}    

03. Which of the following is paramagnetic?
a. O_2^{-}  b. NO^{+}  c. N_2  d. None of these  

06. What is the oxidation number of Cr in [Cr(H_2O)_4Cl_2]Br ? 
a. +2  b. +3  c. -2  d. +6  

07. Which of the following ion shows paramagnetic property?
a. K^{+}  b. Mg^{2+}  c. Ni^{2+}  d. Cu^{+}  

08. If the electron configuration of the d orbital of a metal compound is given, which of the compound will be more paramagnetic?  
a. 2, 2, 2, 2, 1  b. 1, 1, 1, 1, 1  
c. 2, 1, 1, 1, 1  d. 2, 2, 1, 1, 1  

Questions to translate:
01. লেখচিত্রে একটা বস্তুর দূরত্বের (x) সাথে নির্ভরশীল এর উপর ক্রিয়াশীল বলকে (U) দেখানো হয়েছে । বলের একক N এবং দূরত্বের একক m । x\ =\ 0 হতে x\ =\ 6\ m দূরত্বের মধ্যে বস্তুর উপর কৃতকাজের মান কত হবে?
 
a. 4.5\ J 	   b. 13.5\ J 	c. 9.0\ J 	   d. 18\ J

02. F₁ ও F₂ বলদ্বয় দ্বারা 25\ Nm^{-1} ও 16\ Nm^{-1} স্প্রিং ধ্রুবক বিশিষ্ট দুটি স্প্রিংকে সম্প্রসারিত করা হল । যদি স্প্রিং এর সম্প্রসারণে কৃত কাজের পরিমাণ সমান হয় তবে  \frac{F_1}{F_2}\  এর মান কত হবে?
a. \frac{4}{5}        b. \frac{5}{4}       c. \frac{16}{25}      d. \frac{25}{16}

03. চিত্রে, কৃতকাজ কত একক? 
a. 100\pi		b. 50\pi
c. 25\pi		d. 12.5\pi

04. h মিটার উচ্চতা হতে একটি বস্তু নিচে পড়ে গেল । কোথায় তার গতিশক্তি স্থিতিশক্তির অর্ধেক হবে?
a. ভূমি থেকে \frac{2h}{3}\ m উপরে
b. উপর থেকে \frac{2h}{3} নিচের দিকে
c. ভূমি থেকে \frac{2h}{2}\ m উপরে
d. উপর থেকে \frac{2h}{2}\ m নিচের দিকে

05. m\ kg ভরের একটি লোহার বল আনুভূমিক পথে চলমান ছিল । হঠাৎ এটি বিস্ফোরিত হয়ে দুটি মান ভরের টুকরোতে পরিণত হয় । পূর্বে বলটির ভরবেগ ছিল 3p । বিস্ফোরণের পর একটি টুকরো 4p ভরবেগে উপরের দিকে চলতে শুরু করে । বিস্ফোরণের পর সিস্টেমটি কত গতিশক্তি লাভ করে?
a. \frac{25p^2}{m}	    b. 16p²m	c. 41p²m	   d. \frac{73p^2}{m}
 
06. 2\ kW ক্ষমতার একটি পাম্প এর দক্ষতা 74% । এটি 20m উপরে প্রতি সেকেন্ডে কতটুকু পানি তুলতে পারবে?
a. 8\ kg 	  b. 7.55\ kg   c. 12.31\ kg   d. 1.02\ kg 

07. একটি বস্তু সমদ্রুতিতে বৃত্তাকার পথে ঘুরলে-
a. এর উপর কোন কাজ হয় না	   b. ত্বরণ থাকে না
c. বল কাজ করে না 	   d. বেগ অপরিবর্তিত থাকে

08. 20\ m উঁচু দালান থেকে টেনিস বল গড়িয়ে মাটিতে পড়ার সময় বেগ 22\ ms^{-1} হয় । অনুভূমিক বেগ কত?
a. 4.7 ms^{-1}  		b. 7.29 ms^{-1}
c. 11.21\ ms^{-1} 		d. 9.59 ms^{-1}

09.                   উল্লম্বতলে 100\ gm ভরের একটি বস্তুকে ঘুরিয়ে A থেকে B অবস্থানে আনলে
 শক্তির পরিবর্তন কত? [g\ =\ 10\ ms^{-2}]
a. 20J 	  b. 60J     c. 40J 	d. 10J

10. একটি কুয়ার ব্যাসার্ধ 2\ m , গভীরতা 10\ m , এর \frac{3}{4} অংশ পানি দ্বারা পূর্ণ থাকলে একে সম্পূর্ণ পানিশূন্য করতে কী পরিমাণ কাজ করতে হবে?
a. 5.77\ \times\ 10^6\ J   	b. 6\ kJ
c. 2.19\ x\ 10^6\ J   		d. 4.17\ x\ 10^6\ J 
"""

In [61]:
# api.execute_prompt(prompt, timeout=100)
api.execute_prompt("What is the result of 1094*5678?\n response like 'ans = <number>'", timeout=100)

Found 1 response containers


{'prompt': "What is the result of 1094*5678?\n response like 'ans = <number>'",
 'response': {'response': 'ans = 6211732',
  'error': None,
  'id': '2ecb242414e382cc'}}

In [30]:
api.driver.refresh()

In [58]:
res = api.get_last_new_response()
res

Found 3 response containers


{'response': None, 'error': 'no new response', 'id': None}

In [6]:
api.quit_driver()
api = None

In [9]:
import requests
response = requests.get("http://localhost:8000/api/local/?prompt=" + prompt +"&timeout=200")
print(response.json())

{'result': {'prompt': '\ntranslate the questions to english.\nwrite latex code where necessary (i.e. formula of compound [Cr(NH_3)_6]^{3 }). \nwrite all text in code editor \ninsert subscript "_" where needed. (i.e. H_2O)\nTo keep a space between value and unit insert "\\," between. i.e. mass = 400\\,g and velocity = 30\\,kmh^{-1} and (3,\\,0)\n\nwrite a,b,c,d options in 1 line when options lengths are small\nwrite a,b,c,d options in 2 lines (2 per line) when option length are big\n\nsample output format/style to follow:\n01. Which of the following is diamagnetic? \na. [Cr(NH_3)_6]^{3 }\tb. [Ni(CN)_4]^{2-}  \nc. [CuCl_4]^{2-}  \td. [CoF_6]^{2-}    \n\n03. Which of the following is paramagnetic?\na. O_2^{-}  b. NO^{ }  c. N_2  d. None of these  \n\n06. What is the oxidation number of Cr in [Cr(H_2O)_4Cl_2]Br ? \na.  2  b.  3  c. -2  d.  6  \n\n07. Which of the following ion shows paramagnetic property?\na. K^{ }  b. Mg^{2 }  c. Ni^{2 }  d. Cu^{ }  \n\n08. If the electron configuration o